# Amazon ML Challenge 2026: Stage 1 FAISS Retrieval & Gate 1 Audit (GPU-Optimized)
- **GPU-accelerated** dual FAISS IVFFlat Inner Product indexes for Name and Address channels
- S2 on GPU 0, S3 on GPU 1 — each shard ~7.7 GB, fits within 16 GB T4
- Small query batches (16K) to avoid temp-buffer OOM during search
- Top-20 nearest neighbors per route, merged across shards
- Union & deduplication
- Gate 1 Pair Completeness (PC) audit (target >= 0.95)
- Exports `candidates_train.parquet` and `candidate_pairs.tsv`

**Requirements:** Kaggle GPU accelerator (T4 x2) enabled

In [ ]:
!pip install -q faiss-gpu

import os, sys, gc, time
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import pyarrow as pa
import pyarrow.parquet as pq

print("FAISS version:", faiss.__version__)

# ---------- GPU setup ----------
ngpus = faiss.get_num_gpus()
print(f"Available GPUs: {ngpus}")
assert ngpus > 0, "ERROR: No GPU detected! Enable GPU accelerator in Kaggle settings."

# Robust zero-copy loader: reads both standard .npy files and raw np.memmap disk arrays
def load_emb_mmap(filepath, dim=384):
    filepath = Path(filepath)
    try:
        return np.load(filepath, mmap_mode='r')
    except (ValueError, OSError):
        size = filepath.stat().st_size
        n_rows = size // (dim * 2)  # 2 bytes per float16
        return np.memmap(filepath, dtype=np.float16, mode='r', shape=(n_rows, dim))

# Locate input directories
def find_dir_by_file(pattern):
    for p in Path('/kaggle/input').rglob(pattern):
        return p.parent
    return None

PART1_DIR = find_dir_by_file('train_s1_name_emb.npy')
PART2_DIR = find_dir_by_file('train_s3_name_emb.npy')
DATA_DIR  = find_dir_by_file('train_ground_truth.tsv')

if PART1_DIR is None:
    PART1_DIR = Path('/kaggle/input/notebooks/ojassangwai/ml2nbv1/embeddings')
if PART2_DIR is None:
    PART2_DIR = Path('/kaggle/input/notebooks/ojassangwai/ml2nbv2/embeddings')
if DATA_DIR is None:
    DATA_DIR = Path('/kaggle/input/datasets/ojassangwai/mlchallengedata')

print(f"Part 1 (S1+S2): {PART1_DIR}")
print(f"Part 2 (S3):    {PART2_DIR}")
print(f"Data Dir:       {DATA_DIR}")

WORK_DIR = Path('/kaggle/working')
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# HELPER: Build sharded GPU IVFFlat index, search, merge
# ============================================================
#
# MEMORY BUDGET per T4 (16 GB):
#   Index (IVFFlat, 5M x 384 fp32):  ~7.7 GB
#   Temp for search (query_chunk):   ~0.5-1 GB  (scales with batch size)
#   FAISS overhead:                  ~0.5 GB
#   Headroom:                        ~6-7 GB free
#
# V3 failed: single combined index (10.3M) = 15.8 GB -> OOM on transfer
# V5 failed: S2/S3 sharded to 2 GPUs OK, but query_chunk=200K caused
#            ~8 GB temp alloc during .search() -> OOM
# V6 fix:    query_chunk=16384 -> ~650 MB temp, fits easily

dim = 384
top_k = 20
nlist = 4096
nprobe = 64
train_samples = 200000

# CRITICAL: query batch size controls GPU temp memory during search.
# IVFFlat search allocates buffers ~ n_queries * nprobe * sizeof(float) * dim.
# 16K queries * 64 nprobe * 384 * 4B = ~1.5 GB temp (safe with ~8 GB free).
# 200K queries (V5) = ~8 GB temp -> OOM.
query_chunk = 16384


def _build_shard_cpu_index(emb_path, dim, label):
    """Load one shard into RAM, train IVFFlat, add all vectors, return CPU index."""
    mmap = load_emb_mmap(emb_path, dim=dim)
    print(f"  Loading {label} embeddings into RAM: shape {mmap.shape}, dtype {mmap.dtype}")
    t_load = time.time()
    arr = np.asarray(mmap, dtype=np.float32)  # one sequential read + cast
    del mmap
    print(f"    Loaded in {time.time() - t_load:.1f}s")

    n = len(arr)
    n_train = min(train_samples, n)
    rng = np.random.default_rng(42)
    train_idx = rng.choice(n, size=n_train, replace=False)
    train_data = arr[train_idx]

    quantizer = faiss.IndexFlatIP(dim)
    cpu_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

    print(f"  Training IVF quantizer on {n_train:,} random samples ({label})...")
    cpu_index.train(train_data)
    del train_data, train_idx
    gc.collect()

    print(f"  Adding {label} to index ({n:,} vectors, single call)...")
    cpu_index.add(arr)
    del arr
    gc.collect()

    return cpu_index


def build_and_search_gpu(channel_name, s2_emb_path, s3_emb_path, s1_emb_path, n_target):
    """
    Build two IVFFlat GPU indices (S2 on GPU 0, S3 on GPU 1), search S1
    queries against both with small batches, merge to global top_k.
    """
    t0 = time.time()
    print(f"\n{'=' * 50}")
    print(f" {channel_name.upper()} CHANNEL RETRIEVAL (GPU, sharded S2/S3)")
    print(f"{'=' * 50}")

    # --- S2 shard -> GPU 0 ---
    cpu_index_s2 = _build_shard_cpu_index(s2_emb_path, dim, f"S2 {channel_name}")
    n_s2 = cpu_index_s2.ntotal
    print(f"  S2 index built: {n_s2:,} vectors. Transferring to GPU 0...")
    res0 = faiss.StandardGpuResources()
    res0.setTempMemory(1536 * 1024 * 1024)   # 1.5 GB temp (was 256MB)
    gpu_index_s2 = faiss.index_cpu_to_gpu(res0, 0, cpu_index_s2)
    gpu_index_s2.nprobe = nprobe
    del cpu_index_s2
    gc.collect()
    print(f"  S2 on GPU 0: OK")

    # --- S3 shard -> GPU 1 ---
    cpu_index_s3 = _build_shard_cpu_index(s3_emb_path, dim, f"S3 {channel_name}")
    n_s3 = cpu_index_s3.ntotal
    print(f"  S3 index built: {n_s3:,} vectors. Transferring to GPU 1...")
    res1 = faiss.StandardGpuResources()
    res1.setTempMemory(1536 * 1024 * 1024)   # 1.5 GB temp
    gpu_index_s3 = faiss.index_cpu_to_gpu(res1, 1, cpu_index_s3)
    gpu_index_s3.nprobe = nprobe
    del cpu_index_s3
    gc.collect()
    print(f"  S3 on GPU 1: OK")

    print(f"  Both shards ready ({n_s2:,} + {n_s3:,} = {n_s2 + n_s3:,} vectors). "
          f"Searching top-{top_k} per shard (batch={query_chunk:,})...")

    # --- Query S1 embeddings against both shards, merge per row ---
    s1_mmap = load_emb_mmap(s1_emb_path, dim=dim)
    n_s1 = len(s1_mmap)

    all_indices = np.empty((n_s1, top_k), dtype=np.int64)
    all_sims = np.empty((n_s1, top_k), dtype=np.float32)

    t_search = time.time()
    n_batches = (n_s1 + query_chunk - 1) // query_chunk
    for batch_i, start in enumerate(range(0, n_s1, query_chunk)):
        end = min(start + query_chunk, n_s1)
        q_batch = s1_mmap[start:end].astype(np.float32)

        sims_s2, idx_s2 = gpu_index_s2.search(q_batch, top_k)
        sims_s3, idx_s3 = gpu_index_s3.search(q_batch, top_k)
        # S3 local indices -> global index space (S2 comes first in target_ids)
        idx_s3_global = idx_s3 + n_s2

        # Merge top-k from both shards
        merged_idx = np.concatenate([idx_s2, idx_s3_global], axis=1)    # (batch, 2*top_k)
        merged_sims = np.concatenate([sims_s2, sims_s3], axis=1)
        top_order = np.argsort(-merged_sims, axis=1)[:, :top_k]
        row_ar = np.arange(merged_sims.shape[0])[:, None]

        all_indices[start:end] = merged_idx[row_ar, top_order]
        all_sims[start:end] = merged_sims[row_ar, top_order]

        if (batch_i + 1) % 20 == 0 or (batch_i + 1) == n_batches:
            elapsed_s = time.time() - t_search
            qps = end / elapsed_s
            eta = (n_s1 - end) / qps if qps > 0 else 0
            print(f"    Batch {batch_i+1}/{n_batches}: {end:,}/{n_s1:,} queries "
                  f"({qps:,.0f} q/s, ETA {eta:.0f}s)")

    del s1_mmap, gpu_index_s2, gpu_index_s3, res0, res1
    gc.collect()

    elapsed = time.time() - t0
    print(f"  {channel_name} pass complete in {elapsed:.1f}s.")

    return n_s1, all_indices.astype(np.int32), all_sims.astype(np.float16)


In [ ]:
# ============================================================
# LOAD TARGET IDS
# ============================================================

if (PART1_DIR / "train_s2_ids.parquet").exists():
    df_s2_ids = pd.read_parquet(PART1_DIR / "train_s2_ids.parquet")
    df_s3_ids = pd.read_parquet(PART2_DIR / "train_s3_ids.parquet")
    target_ids = np.concatenate([df_s2_ids['entity_id'].values, df_s3_ids['entity_id'].values])
    del df_s2_ids, df_s3_ids
else:
    target_ids = pd.read_parquet(PART1_DIR / "train_target_ids.parquet")['entity_id'].values

n_target = len(target_ids)
print(f"Total target records (S2 + S3): {n_target:,}")
gc.collect()

In [ ]:
# ============================================================
# PASS 1: NAME CHANNEL (GPU)
# ============================================================

n_s1, name_cand_indices, name_cand_sims = build_and_search_gpu(
    channel_name="Name",
    s2_emb_path=PART1_DIR / "train_s2_name_emb.npy",
    s3_emb_path=PART2_DIR / "train_s3_name_emb.npy",
    s1_emb_path=PART1_DIR / "train_s1_name_emb.npy",
    n_target=n_target,
)

In [ ]:
# ============================================================
# PASS 2: ADDRESS CHANNEL (GPU)
# ============================================================

_, addr_cand_indices, addr_cand_sims = build_and_search_gpu(
    channel_name="Address",
    s2_emb_path=PART1_DIR / "train_s2_addr_emb.npy",
    s3_emb_path=PART2_DIR / "train_s3_addr_emb.npy",
    s1_emb_path=PART1_DIR / "train_s1_addr_emb.npy",
    n_target=n_target,
)

In [ ]:
# ============================================================
# PASS 3: UNION, PARQUET STREAMING & GATE 1 AUDIT
# ============================================================

t0 = time.time()
print("\n" + "=" * 50)
print(" PASS 3: UNION, PARQUET STREAMING & GATE 1 AUDIT")
print("=" * 50)

# ---------- Load S1 IDs ----------
if (PART1_DIR / "train_s1_ids.parquet").exists():
    s1_ids = pd.read_parquet(PART1_DIR / "train_s1_ids.parquet")['entity_id'].values
else:
    norm_s1 = find_dir_by_file('train_s1_norm.parquet')
    s1_ids = pd.read_parquet(norm_s1 / "train_s1_norm.parquet")['entity_id'].values

# ---------- Load Ground Truth (vectorized, no iterrows) ----------
gt_path = DATA_DIR / "train_ground_truth.tsv"
print(f"Loading ground truth from {gt_path}...")
df_gt = pd.read_csv(gt_path, sep="\t", dtype=str)

gt_map = {}
total_true_pairs = 0
src_col = df_gt['source1_entity_id'].astype(str).str.strip()
match_col = df_gt['matched_entity_ids'].fillna('').astype(str)

for sid, raw_m in zip(src_col, match_col):
    if raw_m and raw_m != 'nan':
        m_set = {m.strip() for m in raw_m.split(',') if m.strip()}
        gt_map[sid] = m_set
        total_true_pairs += len(m_set)
    else:
        gt_map[sid] = set()

del df_gt, src_col, match_col
gc.collect()
print(f"  Ground truth: {len(gt_map):,} source entities, {total_true_pairs:,} true pairs")

# ---------- Vectorized index-to-ID mapping ----------
print("  Mapping FAISS indices to entity IDs (vectorized)...")

# Clip invalid indices to 0 (they will be filtered by the downstream pipeline)
name_cand_indices_clipped = np.clip(name_cand_indices, 0, n_target - 1)
addr_cand_indices_clipped = np.clip(addr_cand_indices, 0, n_target - 1)

# Vectorized lookup: (n_s1, top_k) -> target entity IDs
name_cand_ids = target_ids[name_cand_indices_clipped]   # shape (n_s1, top_k)
addr_cand_ids = target_ids[addr_cand_indices_clipped]   # shape (n_s1, top_k)

del name_cand_indices_clipped, addr_cand_indices_clipped
gc.collect()

print(f"  Building candidate pairs and streaming to Parquet...")

In [ ]:
# ---------- Streaming merge to Parquet ----------

cand_parquet_path = WORK_DIR / "candidates_train.parquet"
cand_tsv_path = OUTPUT_DIR / "candidate_pairs.tsv"

schema = pa.schema([
    ('s1_id', pa.string()),
    ('candidate_id', pa.string()),
    ('sim_name', pa.float32()),
    ('sim_addr', pa.float32()),
    ('route_name', pa.int8()),
    ('route_addr', pa.int8()),
])

writer = pq.ParquetWriter(cand_parquet_path, schema, compression='snappy')
tsv_file = open(cand_tsv_path, "w", encoding="utf-8")
tsv_file.write("source1_entity_id\tcandidate_entity_ids\n")

covered_true_pairs = 0
total_candidates_generated = 0
cand_counts_sample = []

batch_size_merge = 50000

for b_start in range(0, n_s1, batch_size_merge):
    b_end = min(b_start + batch_size_merge, n_s1)
    batch_n = b_end - b_start

    b_s1_ids = []
    b_cand_ids = []
    b_sim_names = []
    b_sim_addrs = []
    b_r_names = []
    b_r_addrs = []

    # Pre-slice batch arrays for this chunk
    batch_name_ids = name_cand_ids[b_start:b_end]
    batch_addr_ids = addr_cand_ids[b_start:b_end]
    batch_name_sims = name_cand_sims[b_start:b_end]
    batch_addr_sims = addr_cand_sims[b_start:b_end]

    for i_local in range(batch_n):
        i_global = b_start + i_local
        sid = s1_ids[i_global]
        true_set = gt_map.get(sid, set())

        # Merge name + addr candidates
        cand_dict = {}
        n_ids_row = batch_name_ids[i_local]
        n_sims_row = batch_name_sims[i_local]
        for k in range(top_k):
            tid = n_ids_row[k]
            cand_dict[tid] = [float(n_sims_row[k]), 0.0, 1, 0]

        a_ids_row = batch_addr_ids[i_local]
        a_sims_row = batch_addr_sims[i_local]
        for k in range(top_k):
            tid = a_ids_row[k]
            if tid in cand_dict:
                cand_dict[tid][1] = float(a_sims_row[k])
                cand_dict[tid][3] = 1
            else:
                cand_dict[tid] = [0.0, float(a_sims_row[k]), 0, 1]

        # Gate 1 audit
        cand_keys = set(cand_dict.keys())
        if true_set:
            covered_true_pairs += len(cand_keys & true_set)
        if len(cand_counts_sample) < 50000:
            cand_counts_sample.append(len(cand_keys))

        # TSV output
        tsv_file.write(f"{sid}\t{','.join(cand_keys)}\n")

        for cid, (sn, sa, rn, ra) in cand_dict.items():
            b_s1_ids.append(sid)
            b_cand_ids.append(cid)
            b_sim_names.append(sn)
            b_sim_addrs.append(sa)
            b_r_names.append(rn)
            b_r_addrs.append(ra)

    # Flush batch to Parquet
    batch_table = pa.Table.from_arrays([
        pa.array(b_s1_ids, type=pa.string()),
        pa.array(b_cand_ids, type=pa.string()),
        pa.array(b_sim_names, type=pa.float32()),
        pa.array(b_sim_addrs, type=pa.float32()),
        pa.array(b_r_names, type=pa.int8()),
        pa.array(b_r_addrs, type=pa.int8()),
    ], schema=schema)

    writer.write_table(batch_table)
    total_candidates_generated += len(b_s1_ids)
    del batch_table
    gc.collect()

    if (b_start // batch_size_merge) % 10 == 0:
        print(f"    Processed {b_end:,} / {n_s1:,} entities...")

writer.close()
tsv_file.close()
del name_cand_indices, name_cand_sims, addr_cand_indices, addr_cand_sims
del name_cand_ids, addr_cand_ids
gc.collect()

elapsed_merge = time.time() - t0
print(f"\n  Pass 3 complete in {elapsed_merge:.1f}s.")

In [ ]:
# ============================================================
# GATE 1 AUDIT REPORT
# ============================================================

pc = covered_true_pairs / total_true_pairs if total_true_pairs > 0 else 1.0
counts_arr = np.array(cand_counts_sample)

print("\n" + "#" * 60)
print(f" GATE 1 AUDIT REPORT: {'[PASSED]' if pc >= 0.95 else '[WARNING: BELOW 0.95]'}")
print("#" * 60)
print(f" Pair Completeness (PC):     {pc:.4f}  (Target: >= 0.9500)")
print(f" Covered True Pairs:         {covered_true_pairs:,} / {total_true_pairs:,}")
print(f" Total Candidate Pairs:      {total_candidates_generated:,}")
print(f" Mean Candidates / Entity:   {np.mean(counts_arr):.1f}")
print(f" Median Candidates / Entity: {np.median(counts_arr):.1f}")
print(f" P95 Candidates / Entity:    {np.percentile(counts_arr, 95):.1f}")
print(f" Max Candidates / Entity:    {np.max(counts_arr)}")
print(f" Saved: {cand_parquet_path} ({cand_parquet_path.stat().st_size / 1e6:.1f} MB)")
print(f" Saved: {cand_tsv_path} ({cand_tsv_path.stat().st_size / 1e6:.1f} MB)")
print("#" * 60)